# Video 17: Generating Fingerprints for All Molecules
Explanation Script:
"Now we generate Morgan fingerprints for all molecules. This converts each SMILES string into a numerical vector of 1024 bits for machine learning. We'll track progress and display sample fingerprints. Note the deprecation warning - this is just RDKit telling us about a newer method, but our code works fine."

In [1]:
from rdkit import Chem
from rdkit.Chem import AllChem
import pandas as pd
import numpy as np
import time
from IPython.display import display

# Load classified data
df = pd.read_csv('cox2_chembl_classified.csv')
print(f"Generating fingerprints for {len(df)} molecules...")
print("=" * 60)

def smiles_to_fingerprint(smiles, n_bits=1024):
    """Convert SMILES to Morgan fingerprint"""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=n_bits)
            return list(fp)
        else:
            return [0] * n_bits
    except:
        return [0] * n_bits

# Generate fingerprints with progress tracking
fingerprints = []
valid_molecules = 0
invalid_molecules = 0
start_time = time.time()

# Progress tracking dataframe
progress_data = []

for idx, row in df.iterrows():
    smiles = row['canonical_smiles']
    fp = smiles_to_fingerprint(smiles)
    fingerprints.append(fp)
    
    if sum(fp) > 0:
        valid_molecules += 1
    else:
        invalid_molecules += 1
    
    # Record progress every 500 molecules
    if (idx + 1) % 500 == 0:
        elapsed = time.time() - start_time
        progress_data.append({
            'Molecules Processed': idx + 1,
            'Valid': valid_molecules,
            'Invalid': invalid_molecules,
            'Time Elapsed (s)': f"{elapsed:.1f}"
        })

# Display progress
if progress_data:
    print("\nPROGRESS SUMMARY:")
    display(pd.DataFrame(progress_data))

print(f"\n✓ Fingerprint generation complete in {time.time() - start_time:.1f} seconds")
print(f"Valid molecules: {valid_molecules}")
print(f"Invalid molecules: {invalid_molecules}")

# Create fingerprint DataFrame
fp_columns = [f'fp_{i}' for i in range(1024)]
fp_df = pd.DataFrame(fingerprints, columns=fp_columns)

print(f"\nFINGERPRINT MATRIX DIMENSIONS:")
dim_df = pd.DataFrame({
    'Dimension': ['Rows (Molecules)', 'Columns (Bits)'],
    'Size': [fp_df.shape[0], fp_df.shape[1]]
})
display(dim_df)

# Show sample fingerprints for first 5 molecules
print("\nSAMPLE FINGERPRINTS (first 20 bits of first 5 molecules):")
# Create a list of dictionaries for better display
sample_data = []
for i in range(5):
    mol_id = df['molecule_chembl_id'].iloc[i]
    bit_values = fingerprints[i][:20]
    row_data = {'Molecule_ID': mol_id}
    for j, val in enumerate(bit_values):
        row_data[f'bit_{j}'] = val
    sample_data.append(row_data)

sample_fp = pd.DataFrame(sample_data)
display(sample_fp)

# Calculate fingerprint statistics
bit_counts = fp_df.sum()
print("\nFINGERPRINT STATISTICS:")
fp_stats = pd.DataFrame({
    'Metric': ['Average bits per molecule', 'Min bits', 'Max bits', 'Sparsity'],
    'Value': [
        f"{bit_counts.mean():.1f}",
        f"{bit_counts.min()}",
        f"{bit_counts.max()}",
        f"{100 - (bit_counts.mean()/1024*100):.1f}% zeros"
    ]
})
display(fp_stats)

# Combine with original data - using only necessary columns
final_df = pd.concat([
    df[['molecule_chembl_id', 'standard_value', 'pIC50', 'active', 'activity_class']].reset_index(drop=True), 
    fp_df.reset_index(drop=True)
], axis=1)

# Save final dataset
final_df.to_csv('cox2_chembl_fingerprints.csv', index=False)
print(f"\n✓ Final dataset with fingerprints saved to 'cox2_chembl_fingerprints.csv'")
print(f"✓ File contains: {final_df.shape[0]} molecules, {final_df.shape[1]} columns")

# Show final dataset structure - FIXED VERSION
print("\nFINAL DATASET COLUMNS:")
# Get first 5 column names and last 5 column names
first_cols = list(final_df.columns[:5])
last_cols = list(final_df.columns[-5:])

# Create lists of equal length
col_types = ['ID Field', 'Target', 'Target', 'Class', 'Class'] + ['Fingerprint Bits'] * 5
all_cols = first_cols + last_cols

# Create DataFrame for display
col_display = pd.DataFrame({
    'Position': list(range(1, 11)),
    'Column Type': col_types,
    'Column Name': all_cols
})
display(col_display)

# Show first few rows of final dataset
print("\nFIRST 3 ROWS OF FINAL DATASET (showing first 10 columns):")
display(final_df.iloc[:3, :10])

# Show summary of dataset
print("\nDATASET SUMMARY:")
summary_data = {
    'Attribute': ['Total Molecules', 'Total Features', 'Active Compounds', 'Inactive Compounds', 'File Size (MB)'],
    'Value': [
        f"{final_df.shape[0]}",
        f"{final_df.shape[1] - 5} fingerprint bits",
        f"{final_df['active'].sum()}",
        f"{final_df.shape[0] - final_df['active'].sum()}",
        f"{final_df.memory_usage(deep=True).sum() / 1024 / 1024:.1f}"
    ]
}
summary_df = pd.DataFrame(summary_data)
display(summary_df)

Generating fingerprints for 5371 molecules...


[20:54:42] DEPRECATION WARNING: please use MorganGenerator
[20:54:42] DEPRECATION WARNING: please use MorganGenerator
[20:54:42] DEPRECATION WARNING: please use MorganGenerator
[20:54:42] DEPRECATION WARNING: please use MorganGenerator
[20:54:42] DEPRECATION WARNING: please use MorganGenerator
[20:54:42] DEPRECATION WARNING: please use MorganGenerator
[20:54:42] DEPRECATION WARNING: please use MorganGenerator
[20:54:42] DEPRECATION WARNING: please use MorganGenerator
[20:54:42] DEPRECATION WARNING: please use MorganGenerator
[20:54:42] DEPRECATION WARNING: please use MorganGenerator
[20:54:42] DEPRECATION WARNING: please use MorganGenerator
[20:54:42] DEPRECATION WARNING: please use MorganGenerator
[20:54:42] DEPRECATION WARNING: please use MorganGenerator
[20:54:42] DEPRECATION WARNING: please use MorganGenerator
[20:54:42] DEPRECATION WARNING: please use MorganGenerator
[20:54:42] DEPRECATION WARNING: please use MorganGenerator
[20:54:42] DEPRECATION WARNING: please use MorganGenerat


PROGRESS SUMMARY:


[20:54:51] DEPRECATION WARNING: please use MorganGenerator
[20:54:51] DEPRECATION WARNING: please use MorganGenerator
[20:54:51] DEPRECATION WARNING: please use MorganGenerator
[20:54:51] DEPRECATION WARNING: please use MorganGenerator
[20:54:51] DEPRECATION WARNING: please use MorganGenerator
[20:54:51] DEPRECATION WARNING: please use MorganGenerator
[20:54:51] DEPRECATION WARNING: please use MorganGenerator
[20:54:51] DEPRECATION WARNING: please use MorganGenerator
[20:54:51] DEPRECATION WARNING: please use MorganGenerator
[20:54:51] DEPRECATION WARNING: please use MorganGenerator
[20:54:51] DEPRECATION WARNING: please use MorganGenerator
[20:54:51] DEPRECATION WARNING: please use MorganGenerator
[20:54:51] DEPRECATION WARNING: please use MorganGenerator
[20:54:51] DEPRECATION WARNING: please use MorganGenerator
[20:54:51] DEPRECATION WARNING: please use MorganGenerator
[20:54:51] DEPRECATION WARNING: please use MorganGenerator
[20:54:51] DEPRECATION WARNING: please use MorganGenerat

,Molecules Processed,Valid,Invalid,Time Elapsed (s)
0,500,500,0,1.1
1,1000,1000,0,2.3
2,1500,1500,0,3.0
3,2000,2000,0,3.7
4,2500,2500,0,4.3
5,3000,3000,0,5.4
6,3500,3500,0,6.5
7,4000,3999,1,7.2
8,4500,4499,1,7.8
9,5000,4999,1,8.4


...

,Position,Column Type,Column Name
0,1,ID Field,molecule_chembl_id
1,2,Target,standard_value
2,3,Target,pIC50
3,4,Class,active
4,5,Class,activity_class
5,6,Fingerprint Bits,fp_1019
6,7,Fingerprint Bits,fp_1020
7,8,Fingerprint Bits,fp_1021
8,9,Fingerprint Bits,fp_1022
9,10,Fingerprint Bits,fp_1023



FIRST 3 ROWS OF FINAL DATASET (showing first 10 columns):


,molecule_chembl_id,standard_value,pIC50,active,activity_class,fp_0,fp_1,fp_2,fp_3,fp_4
0,CHEMBL100091,60.0,7.22,1,Active,0,0,0,0,0
1,CHEMBL100092,60.0,7.22,1,Active,0,0,0,0,0
2,CHEMBL100156,4000.0,5.40,0,Inactive,0,0,0,0,0



DATASET SUMMARY:


,Attribute,Value
0,Total Molecules,5371
1,Total Features,1024 fingerprint bits
2,Active Compounds,2384
3,Inactive Compounds,2987
4,File Size (MB),42.7


Explanation of Output:
"Each molecule is now represented by 1024 fingerprint bits. The deprecation warnings are normal and don't affect results. On average, each molecule has 27.8 bits set to 1 (97.3% sparsity). The final dataset has 5,371 rows and 1,029 columns: 5 identifier/target columns plus 1024 fingerprint bits. This is now ready for machine learning model training."